# Ad-Streams Propensity Model — Experiment & Register

Model development for the Ad-Streams propensity engine. This notebook is both the
interactive surface and the scheduled job: `retrain_propensity_task` runs it every
Monday at 06:00 UTC via `EXECUTE NOTEBOOK`.

This notebook:
1. Loads engagement features from `dt_user_features` (leakage-free)
2. Races **4 model families** (regularized Logistic Regression, Random Forest, XGBoost, LightGBM) as tracked **ML Experiment** runs
3. Selects a champion by F1, excluding degenerate models that score AUC >= 0.999
4. Registers it under the next free version number and flips the default pointer

Because it reruns on a schedule, run names and version numbers are derived at
runtime rather than hardcoded. The registered model is called by
`dt_user_propensity` in the streaming pipeline.

In [ ]:
from snowflake.snowpark.context import get_active_session
import pandas as pd
import numpy as np
from datetime import datetime, timezone

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

from snowflake.ml.experiment import ExperimentTracking
from snowflake.ml.registry import Registry
from snowflake.ml.model import task

session = get_active_session()
session.use_schema("DEMO_ATAHIR.AD_STREAMS")
print("Account:", session.get_current_account())

## Load features and build the label

**Label:** `CONVERTED = 1` if the user has any conversion.

**Features (10, engagement-only):** click / impression / paid-search counts over 1h, 24h, 7d windows plus 24h event velocity. We deliberately exclude `conversions_total`, `last_conversion_ts`, and `time_since_last_conversion_hrs` to avoid leaking the label.

In [ ]:
FEATURES = [
    "CLICKS_1H", "CLICKS_24H", "CLICKS_7D",
    "IMPRESSIONS_1H", "IMPRESSIONS_24H", "IMPRESSIONS_7D",
    "PAID_SEARCH_1H", "PAID_SEARCH_24H", "PAID_SEARCH_7D",
    "EVENT_VELOCITY_24H",
]

df = session.sql(f"""
    SELECT {', '.join(FEATURES)},
           IFF(conversions_total > 0, 1, 0) AS CONVERTED
    FROM DEMO_ATAHIR.AD_STREAMS.dt_user_features
""").to_pandas()

X = df[FEATURES].astype(float)
y = df["CONVERTED"].astype(int)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)
print(f"Train: {len(X_train)} | Test: {len(X_test)} | Positive rate: {y.mean():.2f}")
X_train.head()

## Experiment: race 4 model families

Each model is a tracked **run** in `AD_PROPENSITY_EXPERIMENT`. Params and metrics are logged so we can compare them side by side in Snowsight (AI & ML » Experiments).

In [ ]:
exp = ExperimentTracking(
    session, database_name="DEMO_ATAHIR", schema_name="ML_EXPERIMENTS"
)
exp.set_experiment("AD_PROPENSITY_EXPERIMENT")

candidates = {
    # Regularized and scaled: strong L2 keeps the marginal response per feature
    # smooth and monotonic, and avoids the degenerate perfect separation an
    # unregularized fit produces on this feature set.
    "logreg_reg":    make_pipeline(StandardScaler(),
                                   LogisticRegression(C=0.05, max_iter=2000)),
    "random_forest": RandomForestClassifier(n_estimators=200, random_state=42),
    "xgboost":       XGBClassifier(n_estimators=200, max_depth=4, eval_metric="logloss"),
    "lightgbm":      LGBMClassifier(n_estimators=200, max_depth=4, verbose=-1),
}

# Runs are namespaced per execution. The weekly retrain task reruns this
# notebook, and reusing bare model names would collide across runs.
RUN_TAG = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
print(f"Run tag: {RUN_TAG}")

results = {}
for name, model in candidates.items():
    with exp.start_run(f"{name}_{RUN_TAG}"):
        model.fit(X_train, y_train)
        proba = model.predict_proba(X_test)[:, 1]
        preds = model.predict(X_test)
        auc = roc_auc_score(y_test, proba)
        acc = accuracy_score(y_test, preds)
        f1  = f1_score(y_test, preds, zero_division=0)
        exp.log_params({k: str(v) for k, v in model.get_params().items()})
        exp.log_metrics({"roc_auc": auc, "accuracy": acc, "f1": f1})
        results[name] = {"model": model, "roc_auc": auc, "accuracy": acc, "f1": f1}
        print(f"{name:14s} AUC={auc:.3f}  ACC={acc:.3f}  F1={f1:.3f}")

In [ ]:
# Compare and pick the champion
leaderboard = pd.DataFrame(
    [{"model": k, **{m: v[m] for m in ("roc_auc", "accuracy", "f1")}} for k, v in results.items()]
).sort_values("roc_auc", ascending=False).reset_index(drop=True)

# Reject suspiciously-perfect models. AUC >= 0.999 here means degenerate perfect
# separation, which gives unstable, non-monotonic coefficients rather than a
# genuinely better model.
eligible = leaderboard[leaderboard["roc_auc"] < 0.999]
if eligible.empty:
    eligible = leaderboard

# Among the eligible models, select by F1 rather than AUC: it balances precision
# against recall on a heavily imbalanced label.
eligible = eligible.sort_values("f1", ascending=False)
champion_name = eligible.iloc[0]["model"]
champion = results[champion_name]["model"]
print(f"Champion (by F1, excluding AUC>=0.999): {champion_name}")
leaderboard

## Register the champion to the Model Registry

The model logs as an **IMMUTABLE** function, so it can be called inside the **incremental** `dt_user_propensity` Dynamic Table without forcing a full refresh. Promoting a new version later is a one-line `default` flip — no pipeline DDL change.

In [ ]:
reg = Registry(session, database_name="DEMO_ATAHIR", schema_name="ML_REGISTRY")

# Derive the next version instead of hardcoding one. This notebook is rerun by
# retrain_propensity_task every Monday, and a fixed version_name would fail on
# the second run because the version already exists.
try:
    existing = reg.get_model("AD_PROPENSITY_MODEL").show_versions()["name"].tolist()
    used = [int(v[1:]) for v in existing if v.upper().startswith("V") and v[1:].isdigit()]
    next_version = f"V{max(used) + 1}" if used else "V1"
except Exception:
    existing, next_version = [], "V1"
print(f"Existing versions: {existing} -> registering {next_version}")

mv = reg.log_model(
    champion,
    model_name="AD_PROPENSITY_MODEL",
    version_name=next_version,
    sample_input_data=X_train,
    conda_dependencies=["scikit-learn", "xgboost", "lightgbm"],
    metrics={
        "roc_auc": float(results[champion_name]["roc_auc"]),
        "accuracy": float(results[champion_name]["accuracy"]),
        "f1": float(results[champion_name]["f1"]),
        "champion": champion_name,
        "training_rows": int(len(X_train)),
        "run_tag": RUN_TAG,
    },
    comment=f"{next_version} champion={champion_name}, selected by F1 across 4 families.",
    task=task.Task.TABULAR_BINARY_CLASSIFICATION,
)
print("Registered:", mv.model_name, mv.version_name)
print("Functions:", [f["name"] for f in mv.show_functions()])

# Promote by flipping the default pointer. dt_user_propensity calls the model by
# name, so the pipeline picks this up on its next refresh with no DDL change.
m = reg.get_model("AD_PROPENSITY_MODEL")
m.default = next_version
print("Default version:", m.default.version_name)

# Smoke-test inference; note the positive-class output column the DT will read
preds = mv.run(X_test.head(5), function_name="predict_proba")
print(preds.columns.tolist())
preds